In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

In [3]:
from src.experiments.prompt import build_system_prompt
from src.lvlms.chat import LVLMChat

In [ ]:
from time import sleep

LLM_JUDGE_PROMPT = """You are an impartial judge in an cooperative object matching game between a director and a matcher. \
In this game, the matcher has to identify 12 objects based on the director's descriptions. Based on the matcher's last response, \
your task is to determine if the matcher has verbally indicated that they have completed their guesses and is ready to submit their answers. \
In other words, you need to decide if the current round of the game is over simply based on the matcher's last reply.

Here is the conversation between the director and the matcher, with the matcher's last reply at the end:

{conversation}

Are you very sure that the current round is over and the matcher is ready to submit their answers? Just say 'YES' or 'NO' and nothing else.
"""


def is_a_round_over(matcher_reply: str, 
                    llm_judge: LVLMChat | None, 
                    conversation: list[str],
                    use_judge: bool) -> bool:
    if '[SUBMIT]' in matcher_reply:
        return True

    if llm_judge is not None and use_judge:
        llm_judge.reset_chat() # no need to keep history for judge
        prompt = LLM_JUDGE_PROMPT.format(conversation=conversation)
        judge_response = llm_judge([prompt])
        print("(==> LLM Judge Response:", judge_response + ")", "\n")
        return judge_response.upper().strip() == 'YES'

    return False


def run_ai_ai_experiment_one_round(director: LVLMChat, 
                                   matcher: LVLMChat,
                                   director_image_fp: str, 
                                   matcher_image_fp: str, 
                                   round_ix: int = 1,
                                   max_num_turns: int = 50, 
                                   llm_judge: LVLMChat | None = None, 
                                   mini_turn_num_to_run_llm_judge: int = 12, 
                                   sleep_time: float = 5) -> None:
    """This function is a proof of concept. It does not record anything.
    The chat history can be found within the director and matcher objects, i.e.,
    director.messages and matcher.messages.

    The matcher's final answers for submission are printed out when the round is over.
    Should be returned or recorded in a real experiment.
    """
    turn_ix = 1
    print(f"{'='*20} Turn#{turn_ix} {'='*20}\n")

    conversation = []

    director_init_msg = director([f"Round {round_ix} Begins.", director_image_fp])
    print("==> Director:", director_init_msg, "\n")

    sleep(sleep_time)

    matcher_reply = matcher([f"Round {round_ix} Begins.", matcher_image_fp, director_init_msg])
    print("==> Matcher:", matcher_reply, "\n")

    conversation.extend([director_init_msg, matcher_reply])

    max_num_turns -= 1
    turn_ix += 1

    while max_num_turns > 0:
        print(f"{'='*20} Turn#{turn_ix} {'='*20}\n")

        director_reply = director(
            [matcher_reply]
        )
        print("==> Director:", director_reply, "\n")

        sleep(sleep_time)

        matcher_reply = matcher(
            [director_reply]
        )
        print("==> Matcher:", matcher_reply, "\n")

        conversation.extend([director_reply, matcher_reply])

        max_num_turns -= 1
        turn_ix += 1

        if is_a_round_over(matcher_reply, llm_judge, conversation,
                           turn_ix >= mini_turn_num_to_run_llm_judge+1):
            
            print(f"{'='*20} Round Over Detected at Turn#{turn_ix-1} {'='*20}\n")
            # the prompt below should be adjusted based on the actual final answer format required
            final_answers = matcher("Summarize the object indices of your final 12 guesses for submission.")
            print("==> Matcher Final Answers for Submission:", final_answers, "\n")
            matcher.messages = matcher.messages[:-1] # Remove last message to reset state
            break


### Test: One Round Only

In [9]:
director_model = "openai/gpt-4o-mini"
matcher_model = "openai/gpt-4o-mini"

director_system_prompt = build_system_prompt("director", "grounded")
matcher_system_prompt = build_system_prompt("matcher", "grounded")
director = LVLMChat(model=director_model, system_prompt=director_system_prompt)
matcher = LVLMChat(model=matcher_model, system_prompt=matcher_system_prompt)

matcher_image_fp = 'data/grid_images/presets5/matcher.png'
director_image_fp = 'data/grid_images/presets5/director_round_1.png'

run_ai_ai_experiment_one_round(
    director=director, 
    matcher=matcher,
    director_image_fp=director_image_fp,
    matcher_image_fp=matcher_image_fp,
    round_ix=1,
    max_num_turns=20,
    llm_judge=None,
    mini_turn_num_to_run_llm_judge=12,
    sleep_time=20
)

Using litellm for model: openai/gpt-4o-mini
Using litellm for model: openai/gpt-4o-mini
==================== Turn#1 ====================



==> Director: Great! Let's get started.

Basket 1 is shaped like a small animal, specifically a rabbit. It has a rounded body and a lid that resembles the rabbit's head, with two ears on top. The color is a light brown, and the texture is smooth. Can you find it? 

==> Matcher: I see it! The basket shaped like a rabbit with a rounded body and ears on top. What's next? [PLACE:1,1] 

==================== Turn#2 ====================

==> Director: Awesome! Moving on to basket 2.

Basket 2 is a medium-sized, round basket with a flat bottom. It has two curved handles on the sides and is made of a light brown woven material. The weaving pattern is tight and uniform. Can you locate it? 

==> Matcher: I see it! The medium-sized round basket with two curved handles and a flat bottom. What's next? [PLACE:5,2] 

==================== Turn#3 ====================

==> Director: Great job! Now, let's move to basket 3.

Basket 3 is a tall and narrow basket with a cylindrical shape. It has a smooth, li

KeyboardInterrupt: 

In [ ]:
## use judge

director_model = "openai/gpt-5-nano"
matcher_model = "openai/gpt-5-nano"

director_system_prompt = build_system_prompt("director", "grounded")
matcher_system_prompt = build_system_prompt("matcher", "grounded")

director = LVLMChat(model=director_model, system_prompt=director_system_prompt)
matcher = LVLMChat(model=matcher_model, system_prompt=matcher_system_prompt)
judge = LVLMChat(model="Qwen/Qwen3-VL-4B-Instruct")

matcher_image_fp = 'data/grid_images/presets5/matcher.png'
director_image_fp = 'data/grid_images/presets5/director_round_1.png'

run_ai_ai_experiment_one_round(
    director=director, 
    matcher=matcher,
    director_image_fp=director_image_fp,
    matcher_image_fp=matcher_image_fp,
    round_ix=1,
    max_num_turns=20,
    llm_judge=judge,
    mini_turn_num_to_run_llm_judge=12,
    sleep_time=30
)

Using local_via_openai for model: Qwen/Qwen3-VL-4B-Instruct
Using local_via_openai for model: Qwen/Qwen3-VL-4B-Instruct
Using local_via_openai for model: Qwen/Qwen3-VL-4B-Instruct
==================== Turn#1 ====================

==> Director: Basket 1 is a wicker basket shaped like a rabbit, with a rounded body, small ears, and a stitched nose. It’s light brown with a slightly textured weave. The basket has a lid that fits snugly on top. 

Let me know when you’ve found it! 

==> Matcher: I see it! The rabbit-shaped wicker basket with a rounded body, small ears, stitched nose, and snug-fitting lid. It’s light brown with a textured weave. [PLACE:16,1] 

==================== Turn#2 ====================

==> Director: Great! Moving to Basket 2 — this one is a round, deep-woven basket with a single, thick, dark brown handle arching over the top. The sides are slightly curved, and the weave is tight and uniform. It’s the only one with a handle that’s not attached to the sides but loops over

### Different prompts

In [ ]:
## use judge

director_model = "openai/gpt-4o-mini"
matcher_model = "openai/gpt-4o-mini"

director_system_prompt = build_system_prompt("director", "minimal")
matcher_system_prompt = build_system_prompt("matcher", "minimal")

director = LVLMChat(model=director_model, system_prompt=director_system_prompt)
matcher = LVLMChat(model=matcher_model, system_prompt=matcher_system_prompt)
judge = LVLMChat(model="Qwen/Qwen3-VL-4B-Instruct")

matcher_image_fp = 'data/grid_images/presets5/matcher.png'
director_image_fp = 'data/grid_images/presets5/director_round_1.png'

run_ai_ai_experiment_one_round(
    director=director, 
    matcher=matcher,
    director_image_fp=director_image_fp,
    matcher_image_fp=matcher_image_fp,
    round_ix=1,
    max_num_turns=20,
    llm_judge=judge,
    mini_turn_num_to_run_llm_judge=12
)

In [ ]:
## use judge

director_model = "openai/gpt-4o-mini"
matcher_model = "openai/gpt-4o-mini"

director_system_prompt = build_system_prompt("director", "detailed")
matcher_system_prompt = build_system_prompt("matcher", "detailed")

director = LVLMChat(model=director_model, system_prompt=director_system_prompt)
matcher = LVLMChat(model=matcher_model, system_prompt=matcher_system_prompt)
judge = LVLMChat(model="Qwen/Qwen3-VL-4B-Instruct")

matcher_image_fp = 'data/grid_images/presets5/matcher.png'
director_image_fp = 'data/grid_images/presets5/director_round_1.png'

run_ai_ai_experiment_one_round(
    director=director, 
    matcher=matcher,
    director_image_fp=director_image_fp,
    matcher_image_fp=matcher_image_fp,
    round_ix=1,
    max_num_turns=20,
    llm_judge=judge,
    mini_turn_num_to_run_llm_judge=12
)

### Test: Four Rounds

In [ ]:
director_model = "openai/gpt-4o-mini"
matcher_model = "openai/gpt-4o-mini"

director_system_prompt = build_system_prompt("director", "grounded")
matcher_system_prompt = build_system_prompt("matcher", "grounded")

director = LVLMChat(model=director_model, system_prompt=director_system_prompt)
matcher = LVLMChat(model=matcher_model, system_prompt=matcher_system_prompt)
judge = LVLMChat(model="Qwen/Qwen3-VL-4B-Instruct")


# the maximum number of turns per round
max_num_turns = 20

for round_ix in range(1, 5):
    print(f"{'='*50} Starting Round {round_ix} {'='*50}\n")

    matcher_image_fp = 'data/grid_images/presets5/matcher.png'
    director_image_fp = f'data/grid_images/presets5/director_round_{round_ix}.png'

    run_ai_ai_experiment_one_round(
        director=director,
        matcher=matcher,
        director_image_fp=director_image_fp,
        matcher_image_fp=matcher_image_fp,
        max_num_turns=max_num_turns,
        round_ix=round_ix,
        llm_judge=judge,
        mini_turn_num_to_run_llm_judge=12
    )

Using local_via_openai for model: Qwen/Qwen3-VL-4B-Instruct
Using local_via_openai for model: Qwen/Qwen3-VL-4B-Instruct
Using local_via_openai for model: Qwen/Qwen3-VL-4B-Instruct
================================================== Starting Round 1 ==================================================

==================== Turn#1 ====================

==> Director: Basket 1 is a wicker basket shaped like a rabbit, with a rounded body, small ears, and a stitched nose. It’s light brown with a slightly textured weave. The basket has a lid that fits snugly on top. 

Let me know when you’ve found it! 

==> Matcher: I see it! The rabbit-shaped wicker basket with a rounded body, small ears, stitched nose, and snug-fitting lid. It’s light brown with a textured weave. [PLACE:16,1] 

==================== Turn#2 ====================

==> Director: Great! Moving to Basket 2 — this one is a round, dark brown wicker basket with a single, thick, arched handle on top. The weave is tight and uniform, and i